In [1]:
import pandas as pd
import sqlite3
from datetime import datetime

In [3]:
# Payment backend (engine) log extract, all providers
df_payment_engine_log = pd.read_csv("raw/payment_engine_log.csv") 
df_payment_engine_log.head()

,txn_id,order_id,operation_type,status,psp,psp_reference,sku,country,currency,amount_local,fx_rate_applied,fx_date_applied,amount_usd,created_at_utc,captured_at_utc
0,TXN-107955,ORD-507781,SALE,settled,dlocal,DL-90001309,sub_monthly,MX,MXN,129.00,0.055082,2026-05-29,7.11,2026-05-31T23:59:50Z,2026-06-01T00:05:00Z
1,TXN-100884,ORD-500884,SALE,settled,paypal_us,7FTLSR4XAT5TKVLKF,sub_monthly,US,USD,9.99,1.000000,2026-06-01,9.99,2026-06-01T07:03:11Z,2026-06-01T07:12:11Z
2,TXN-106445,ORD-506445,SALE,settled,dlocal,DL-90000750,sub_monthly,CO,COP,29900.00,0.000251,2026-06-01,7.50,2026-06-01T07:11:11Z,2026-06-01T08:27:11Z
3,TXN-106099,ORD-506099,SALE,settled,dlocal,DL-90000404,sub_monthly,MX,MXN,129.00,0.055014,2026-06-01,7.10,2026-06-01T07:11:24Z,2026-06-01T08:17:24Z
4,TXN-105905,ORD-505905,SALE,settled,dlocal,DL-90000210,sub_quarterly,BR,BRL,89.90,0.184519,2026-06-01,16.59,2026-06-01T07:12:27Z,2026-06-01T08:53:27Z


In [20]:
# PayPal US account activity export
df_paypal_us_activity_20260601_20260703 = pd.read_csv("raw/paypal_us_activity_20260601_20260703.csv") 
df_paypal_us_activity_20260601_20260703.columns = df_paypal_us_activity_20260601_20260703.columns.str.lower().str.replace(" ", "_")
df_paypal_us_activity_20260601_20260703.head()


,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id
0,06/01/2026,07:12:11,GMT,Taylor Brooks,7FTLSR4XAT5TKVLKF,NaN,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-500884
1,06/01/2026,07:54:32,GMT,Morgan Hayes,7R82A7SLSXJAD72J1,NaN,Website Payment,Completed,USD,59.99,-2.58,57.41,ORD-500132
2,06/01/2026,07:59:18,GMT,Drew Bennett,S1VNZT7ACC5E6HNYD,NaN,Website Payment,Completed,USD,59.99,-2.58,57.41,ORD-500591
3,06/01/2026,08:03:29,GMT,Riley Carter,BTDR7VQTQ8G4GTMGS,NaN,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-500479
4,06/01/2026,08:16:21,GMT,Casey Nguyen,NFQZ142PMP4P1BC2Q,NaN,Website Payment,Completed,USD,24.99,-1.36,23.63,ORD-500005


In [21]:
# PayPal EU account activity export
# semicolon, cp1252 (Windows-1252)
df_paypal_eu_activity_20260601_20260703 = pd.read_csv("raw/paypal_eu_activity_20260601_20260703.csv", sep=";", encoding="cp1252") 
df_paypal_eu_activity_20260601_20260703.columns = df_paypal_eu_activity_20260601_20260703.columns.str.lower().str.replace(" ", "_")
df_paypal_eu_activity_20260601_20260703.head()

,date,time,time_zone,name,transaction_id,reference_txn_id,type,status,currency,gross,fee,net,invoice_id
0,06/03/2026,12:20:00,GMT,Jürgen Müller,N2D9QMQNH8VUEES5X,NaN,Website Payment,Completed,EUR,"9,99","-0,64","9,35",ORD-507782
1,01/06/2026,08:17:07,GMT,Céline Marchand,ZEW9RTFHKTBFQT5E3,NaN,Website Payment,Completed,GBP,"8,99","-0,56","8,43",ORD-502669
2,01/06/2026,08:35:02,GMT,José García,0DRQ0928NVSA52MA3,NaN,Website Payment,Denied,GBP,"4,49","0,00","0,00",ORD-503463
3,01/06/2026,08:48:13,GMT,Nuño Ibáñez,P54742W2ULA8YDR0L,NaN,Website Payment,Completed,EUR,"4,99","-0,49","4,50",ORD-502556
4,01/06/2026,09:07:25,GMT,Søren Kjær,JR1SUR8G07W17ZPNR,NaN,Website Payment,Completed,EUR,"4,99","-0,49","4,50",ORD-502537


In [22]:
# Adyen payment accounting report
df_adyen_payment_accounting_20260601_20260703 = pd.read_csv("raw/adyen_payment_accounting_20260601_20260703.csv") 
df_adyen_payment_accounting_20260601_20260703.columns = df_adyen_payment_accounting_20260601_20260703.columns.str.lower().str.replace(" ", "_")
df_adyen_payment_accounting_20260601_20260703.head()

,company_account,merchant_account,psp_reference,merchant_reference,record_type,creation_date,timezone,gross_currency,gross_debit,gross_credit,commission,markup,net_currency,net_debit,net_credit,batch_number
0,LumoGroup,LumoAPP_US,8620000000000623,ORD-504573,Authorised,2026-06-01 09:13:28,Europe/Amsterdam,USD,NaN,24.99,NaN,NaN,USD,NaN,NaN,23
1,LumoGroup,LumoECOM_EU,8620000000000500,ORD-504450,Authorised,2026-06-01 09:15:43,Europe/Amsterdam,EUR,NaN,9.99,NaN,NaN,EUR,NaN,NaN,23
2,LumoGroup,LumoECOM_EU,8620000000000115,ORD-504065,Authorised,2026-06-01 09:18:57,Europe/Amsterdam,EUR,NaN,24.99,NaN,NaN,EUR,NaN,NaN,23
3,LumoGroup,LumoAPP_US,8620000000000623,ORD-504573,Settled,2026-06-01 09:36:28,Europe/Amsterdam,USD,NaN,24.99,0.47,0.15,USD,NaN,24.37,23
4,LumoGroup,LumoAPP_US,8620000000000575,ORD-504525,Authorised,2026-06-01 09:41:36,Europe/Amsterdam,USD,NaN,9.99,NaN,NaN,USD,NaN,NaN,23


In [8]:
# dLocal transactions export
df_dlocal_transactions_20260601_20260703 = pd.read_csv("raw/dlocal_transactions_20260601_20260703.csv") 
df_dlocal_transactions_20260601_20260703.head()

,transaction_id,invoice_id,transaction_type,status,created_at_utc,country,currency,currency_exponent,local_amount,fx_rate,usd_amount,fee_usd
0,DL-90001309,ORD-507781,PAYMENT,PAID,2026-06-01 00:05:00,MX,MXN,2,12900,18.1548,7.11,0.32
1,DL-90001242,ORD-506937,PAYMENT,REJECTED,2026-06-01 07:25:52,CO,COP,2,19990000,3984.0637,50.17,2.26
2,DL-90000404,ORD-506099,PAYMENT,PAID,2026-06-01 08:17:24,MX,MXN,2,12900,18.1772,7.10,0.32
3,DL-90000750,ORD-506445,PAYMENT,PAID,2026-06-01 08:27:11,CO,COP,2,2990000,3984.0637,7.50,0.34
4,DL-90001024,ORD-506719,PAYMENT,PAID,2026-06-01 08:29:13,BR,BRL,2,3490,5.4195,6.44,0.29


In [23]:
# Google Play earnings report, June
# line 1 is a report header, data starts on line 2
df_google_play_earnings_202606 = pd.read_csv("raw/google_play_earnings_202606.csv", skiprows=1, header=0)
df_google_play_earnings_202606.columns = df_google_play_earnings_202606.columns.str.lower().str.replace(" ", "_").str.replace(r"[()]", "", regex=True)
df_google_play_earnings_202606.head()

# Earnings report for Lumo Wellness (merchant account: LumoPlayMain). All dates and times are in America/Los_Angeles.

,transaction_date,transaction_time,transaction_type,product_id,buyer_country,buyer_currency,amount_buyer_currency,currency_conversion_rate,merchant_currency,amount_merchant_currency
0,"Jun 1, 2026",00:28:46,Charge,sub_annual,US,USD,59.99,1.0,USD,59.99
1,"Jun 1, 2026",00:28:46,Google fee,sub_annual,US,USD,-9.00,1.0,USD,-9.00
2,"Jun 1, 2026",00:53:08,Charge,sub_monthly,US,USD,9.99,1.0,USD,9.99
3,"Jun 1, 2026",00:53:08,Google fee,sub_monthly,US,USD,-1.50,1.0,USD,-1.50
4,"Jun 1, 2026",01:35:01,Charge,sub_monthly,US,USD,9.99,1.0,USD,9.99


In [24]:
# Google Play earnings report, July 1–3
df_google_play_earnings_202607_partial = pd.read_csv("raw/google_play_earnings_202607_partial.csv", skiprows=1, header=0) 
df_google_play_earnings_202607_partial.columns = df_google_play_earnings_202607_partial.columns.str.lower().str.replace(" ", "_").str.replace(r"[()]", "", regex=True)
df_google_play_earnings_202607_partial.head()

# Earnings report for Lumo Wellness (merchant account: LumoPlayMain). All dates and times are in America/Los_Angeles.

,transaction_date,transaction_time,transaction_type,product_id,buyer_country,buyer_currency,amount_buyer_currency,currency_conversion_rate,merchant_currency,amount_merchant_currency
0,"Jul 1, 2026",02:40:10,Charge refund,sub_monthly,DE,EUR,-9.99,1.085329,USD,-10.84
1,"Jul 1, 2026",03:50:07,Charge,sub_monthly,DE,EUR,9.99,1.085329,USD,10.84
2,"Jul 1, 2026",03:50:07,Google fee,sub_monthly,DE,EUR,-1.50,1.085329,USD,-1.63
3,"Jul 1, 2026",05:43:32,Charge,sub_monthly,US,USD,9.99,1.000000,USD,9.99
4,"Jul 1, 2026",05:43:32,Google fee,sub_monthly,US,USD,-1.50,1.000000,USD,-1.50


In [18]:
# Daily FX rates to USD (usd_rate = USD per 1 unit of currency)
df_fx_rates = pd.read_csv("raw/fx_rates.csv") 
df_fx_rates.head()

,rate_date,currency,usd_rate,published_at
0,2026-05-25,ARS,0.000839,2026-05-25T16:00:00Z
1,2026-05-25,BRL,0.184871,2026-05-25T16:00:00Z
2,2026-05-25,CLP,0.001060,2026-05-25T16:00:00Z
3,2026-05-25,COP,0.000248,2026-05-25T16:00:00Z
4,2026-05-25,EUR,1.093525,2026-05-25T16:00:00Z


In [19]:
# Contracted provider fees, where we have a contract on file
df_fee_schedule = pd.read_csv("raw/fee_schedule.csv") 
df_fee_schedule.head()

,psp,account,valid_from,percent_fee,fixed_fee,fixed_fee_currency
0,adyen,LumoECOM_EU,2026-01-01,2.5,0.0,NaN
1,adyen,LumoAPP_US,2026-01-01,2.5,0.0,NaN
2,google_play,LumoPlayMain,2026-01-01,15.0,0.0,NaN


In [25]:
# connect to a database
conn = sqlite3.connect("db_simple_life.db")

df_payment_engine_log.to_sql('payment_engine_log', conn, if_exists="replace", index=False)

df_paypal_us_activity_20260601_20260703.to_sql('paypal_us_activity_20260601_20260703', conn, if_exists="replace", index=False)
df_paypal_eu_activity_20260601_20260703.to_sql('paypal_eu_activity_20260601_20260703', conn, if_exists="replace", index=False)
df_adyen_payment_accounting_20260601_20260703.to_sql('adyen_payment_accounting_20260601_20260703', conn, if_exists="replace", index=False)
df_dlocal_transactions_20260601_20260703.to_sql('dlocal_transactions_20260601_20260703', conn, if_exists="replace", index=False)
df_google_play_earnings_202606.to_sql('google_play_earnings_202606', conn, if_exists="replace", index=False)
df_google_play_earnings_202607_partial.to_sql('google_play_earnings_202607_partial', conn, if_exists="replace", index=False)
df_fx_rates.to_sql('fx_rates', conn, if_exists="replace", index=False)
df_fee_schedule.to_sql('fee_schedule', conn, if_exists="replace", index=False)


3

In [50]:
query = """
SELECT *
FROM payment_engine_log -- +
-- FROM paypal_us_activity_20260601_20260703 -- +
-- FROM paypal_eu_activity_20260601_20260703 -- +
-- FROM adyen_payment_accounting_20260601_20260703 -- +
-- FROM dlocal_transactions_20260601_20260703 -- +
-- FROM google_play_earnings_202606 -- +
-- FROM google_play_earnings_202607_partial -- +
-- FROM fx_rates -- +
-- FROM fee_schedule -- +
LIMIT 100;
"""

In [51]:
result = pd.read_sql(query, conn)
result

,txn_id,order_id,operation_type,status,psp,psp_reference,sku,country,currency,amount_local,fx_rate_applied,fx_date_applied,amount_usd,created_at_utc,captured_at_utc
0,TXN-107955,ORD-507781,SALE,settled,dlocal,DL-90001309,sub_monthly,MX,MXN,129.00,0.055082,2026-05-29,7.11,2026-05-31T23:59:50Z,2026-06-01T00:05:00Z
1,TXN-100884,ORD-500884,SALE,settled,paypal_us,7FTLSR4XAT5TKVLKF,sub_monthly,US,USD,9.99,1.000000,2026-06-01,9.99,2026-06-01T07:03:11Z,2026-06-01T07:12:11Z
2,TXN-106445,ORD-506445,SALE,settled,dlocal,DL-90000750,sub_monthly,CO,COP,29900.00,0.000251,2026-06-01,7.50,2026-06-01T07:11:11Z,2026-06-01T08:27:11Z
3,TXN-106099,ORD-506099,SALE,settled,dlocal,DL-90000404,sub_monthly,MX,MXN,129.00,0.055014,2026-06-01,7.10,2026-06-01T07:11:24Z,2026-06-01T08:17:24Z
4,TXN-105905,ORD-505905,SALE,settled,dlocal,DL-90000210,sub_quarterly,BR,BRL,89.90,0.184519,2026-06-01,16.59,2026-06-01T07:12:27Z,2026-06-01T08:53:27Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,TXN-104954,ORD-504954,SALE,settled,adyen,8620000000001004,sub_monthly,DE,EUR,9.99,1.091728,2026-06-01,10.91,2026-06-01T11:25:59Z,2026-06-01T12:18:59Z
96,TXN-104394,ORD-504394,SALE,settled,adyen,8620000000000444,sub_monthly,FR,EUR,9.99,1.091728,2026-06-01,10.91,2026-06-01T11:31:13Z,2026-06-01T12:32:13Z
97,TXN-100737,ORD-500737,SALE,settled,paypal_us,09SFGAHY25HP867MR,sub_monthly,US,USD,9.99,1.000000,2026-06-01,9.99,2026-06-01T11:31:48Z,2026-06-01T12:41:48Z
98,TXN-103713,ORD-503713,SALE,settled,paypal_eu,LHV6U2FAGUQF9VL5W,sub_quarterly,DE,EUR,24.99,1.091728,2026-06-01,27.28,2026-06-01T11:32:56Z,2026-06-01T12:46:56Z
